# AI text detection using Qwen3-Embedding-0.6B

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

OUTPUT_DIR='/content/drive/MyDrive/deep_learning/progetto_deep_twitter/embeddings_qwen_06'

## Cella 3: Caricamento dataset

In [ ]:
from datasets import load_dataset
import pandas as pd

dataset = load_dataset('srikanthgali/ai-text-detection-pile-cleaned')

df_train = pd.DataFrame(dataset['train'])
print(df_train.head())
print(f"Colonne: {df_train.columns.tolist()}")
print(f"Totale campioni: {sum(len(dataset[s]) for s in dataset):,}")

# Controlla la distribuzione delle classi
label_col = 'label' if 'label' in df_train.columns else 'generated'
print(f"\nDistribuzione '{label_col}':")
print(df_train[label_col].value_counts(normalize=True))

## Creazione e save degli embedding

In [ ]:
import numpy as np
import os
from tqdm.auto import tqdm

# ── Parametri ──────────────────────────────────────────────────────────────
BATCH_SIZE = 64        # riduci a 32 se vai in OOM
TEXT_COL   = 'text'    # colonna del testo (adatta se serve)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Funzione generica per uno split ────────────────────────────────────────
def encode_and_save(split_name: str):
    df = pd.DataFrame(dataset[split_name])
    texts = df[TEXT_COL].fillna("").tolist()

    all_embeddings = []

    for i in tqdm(range(0, len(texts), BATCH_SIZE),
                  desc=f"Encoding {split_name}"):
        batch = texts[i : i + BATCH_SIZE]
        emb   = model.encode(
            batch,
            batch_size=BATCH_SIZE,
            show_progress_bar=False,
            normalize_embeddings=True,   # cosine-ready
            convert_to_numpy=True
        )
        all_embeddings.append(emb)

    embeddings = np.vstack(all_embeddings)          # (N, D)
    labels     = df['label' if 'label' in df.columns else 'generated'].values

    # Salvataggio
    np.save(os.path.join(OUTPUT_DIR, f"{split_name}_embeddings.npy"), embeddings)
    np.save(os.path.join(OUTPUT_DIR, f"{split_name}_labels.npy"),     labels)

    print(f"[{split_name}] embeddings shape: {embeddings.shape}  →  salvato in {OUTPUT_DIR}")
    return embeddings, labels

# ── Esegui per ogni split disponibile ─────────────────────────────────────
results = {}
for split in dataset.keys():          # 'train', 'test', 'validation' …
    emb, lab = encode_and_save(split)
    results[split] = {"embeddings": emb, "labels": lab}

print("\n✅ Tutti gli embedding sono stati salvati.")